# Generate Single Stagger Radar Dataset With Error Levels 1, 2, 3

This notebook generates synthetic PRI datasets for one stagger radar.

The output is saved under `datasets/single_stagger_1000x128_<error_level>/`.

## Imports

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml

# Error Level Definitions

In [3]:
ERROR_LEVELS = {
    "clean": {
        "poi": 1.0,
        "multipath_prob": 0.0,
        "aoa_drift_rate": 0.0,
        "toa_noise_std": 0.0,
        "freq_noise_std": 0.0,
        "pw_noise_std": 0.0,
        "amp_noise_std": 0.0,
        "aoa_noise_std": 0.0,
        "global_drop_rate": 0.0,
        "noise_rate": 0.0,
        "quantize": False,
    },
    "error_lv1": {
        "poi": 0.995,
        "multipath_prob": 0.0,
        "aoa_drift_rate": 0.0,
        "toa_noise_std": 0.1,
        "freq_noise_std": 0.1,
        "pw_noise_std": 0.01,
        "amp_noise_std": 0.2,
        "aoa_noise_std": 0.05,
        "global_drop_rate": 0.0,
        "noise_rate": 1.0,
        "quantize": False,
    },
    "error_lv2": {
        "poi": 0.99,
        "multipath_prob": 0.01,
        "aoa_drift_rate": 0.2,
        "toa_noise_std": 0.5,
        "freq_noise_std": 0.5,
        "pw_noise_std": 0.03,
        "amp_noise_std": 0.5,
        "aoa_noise_std": 0.2,
        "global_drop_rate": 0.005,
        "noise_rate": 5.0,
        "quantize": False,
    },
    "error_lv3": {
        "poi": 0.98,
        "multipath_prob": 0.03,
        "aoa_drift_rate": 0.5,
        "toa_noise_std": 1.0,
        "freq_noise_std": 1.0,
        "pw_noise_std": 0.05,
        "amp_noise_std": 1.0,
        "aoa_noise_std": 0.5,
        "global_drop_rate": 0.01,
        "noise_rate": 10.0,
        "quantize": False,
    },
}

# Paths

In [4]:
PROJECT_ROOT = Path("..").resolve()

RADAR_TYPE = "stagger"
SEQUENCE_COUNT = 1000
PULSES_PER_SEQUENCE = 128
RANDOM_SEED = 42

GENERATOR_DIR = PROJECT_ROOT / "synthetic data generation"
DATASET_ROOT = PROJECT_ROOT / "datasets"

sys.path.append(str(GENERATOR_DIR))
from radar_generator import Environment, StaggerRadar

RADAR_CLASS = StaggerRadar

PROJECT_ROOT, GENERATOR_DIR

(WindowsPath('C:/Users/ciftc/OneDrive/Desktop/homeworks/4. year/1/EE 493/repo/Bitirme'),
 WindowsPath('C:/Users/ciftc/OneDrive/Desktop/homeworks/4. year/1/EE 493/repo/Bitirme/synthetic data generation'))

## Dataset Configuration

This dataset contains one stagger radar. Its PRI repeats the pattern
850, 950, 1050, 1150 microseconds, with a mean of 1000 microseconds.

The clean dataset retains this stagger pattern but has no additional
observation errors. Other error levels use the settings defined above.

Outputs will go under `datasets/single_stagger/<error_level>/`.

In [5]:
RADAR_ID = "R_STAGGER"
PRI_US = 1000.0
STAGGER_PRIS_US = [850.0, 950.0, 1050.0, 1150.0]

PRI_ARGUMENTS = {"stagger_pris": STAGGER_PRIS_US}
PRI_METADATA = {
    "stagger_pris_us": STAGGER_PRIS_US,
    "mean_pri_us": PRI_US,
}

FREQ_MEAN_MHZ = 9000.0
PW_MEAN_US = 1.5
AMP_MEAN_DBM = -55.0
AOA_MEAN_DEG = 45.0

# Save CSV function definition

In [6]:
def save_individual_sequence_csvs(split_df, split_dir):
    for sequence_id, seq_df in split_df.groupby("sequence_id"):
        seq_df = seq_df.sort_values("pulse_index")
        seq_df.to_csv(split_dir / f"{sequence_id}.csv", index=False)

## Generate Pulse Sequences, Save Sequences.csv and Split Csv Files

Each `sequence_id` represents one independent radar pulse train.

The sequences should not be concatenated as one long continuous signal.
Models and algorithms should process each `sequence_id` separately.

In [ ]:
for LEVEL, cfg in ERROR_LEVELS.items():
    DATASET_NAME = f"single_{RADAR_TYPE}_1000x128_{LEVEL}"
    DATASET_DIR = DATASET_ROOT  / "single_radar" / "fixed_profile" / f"single_{RADAR_TYPE}" / DATASET_NAME

    TRAIN_DIR = DATASET_DIR / "train"
    VAL_DIR = DATASET_DIR / "val"
    TEST_DIR = DATASET_DIR / "test"

    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    VAL_DIR.mkdir(parents=True, exist_ok=True)
    TEST_DIR.mkdir(parents=True, exist_ok=True)

    all_sequences = []

    for i in range(SEQUENCE_COUNT):
        sequence_id = f"seq_{i:04d}"
        env = Environment()

        radar = RADAR_CLASS(
            radar_id=RADAR_ID,
            **PRI_ARGUMENTS,
            freq_mean=FREQ_MEAN_MHZ,
            freq_mode="constant",
            freq_hopping_values=None,
            poi=cfg["poi"],
            multipath_prob=cfg["multipath_prob"],
            multipath_delay_mean=1.5,
            multipath_delay_std=0.2,
            pw_mean=PW_MEAN_US,
            amp_mean=AMP_MEAN_DBM,
            aoa_mean=AOA_MEAN_DEG,
            scan_period=0.0,
            aoa_drift_rate=cfg["aoa_drift_rate"],
            stretch_pw_on_multipath=False,
        )
        env.add_radar(radar)

        df_seq = env.generate(
            duration=PULSES_PER_SEQUENCE * PRI_US,
            toa_noise_std=cfg["toa_noise_std"],
            freq_noise_std=cfg["freq_noise_std"],
            pw_noise_std=cfg["pw_noise_std"],
            amp_noise_std=cfg["amp_noise_std"],
            aoa_noise_std=cfg["aoa_noise_std"],
            global_drop_rate=cfg["global_drop_rate"],
            noise_rate=cfg["noise_rate"],
            quantize=cfg["quantize"],
        )

        df_seq = df_seq.sort_values("TOA").reset_index(drop=True)
        df_seq["sequence_id"] = sequence_id
        df_seq["pulse_index"] = np.arange(len(df_seq))
        df_seq["radar_type"] = RADAR_TYPE
        df_seq["error_level"] = LEVEL

        all_sequences.append(df_seq)

    df = pd.concat(all_sequences, ignore_index=True)

    sequence_ids = sorted(df["sequence_id"].unique())
    rng = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(sequence_ids)

    train_ids = set(sequence_ids[:700])
    val_ids = set(sequence_ids[700:800])
    test_ids = set(sequence_ids[800:1000])

    train_df = df[df["sequence_id"].isin(train_ids)].copy()
    val_df = df[df["sequence_id"].isin(val_ids)].copy()
    test_df = df[df["sequence_id"].isin(test_ids)].copy()

    train_df.to_csv(TRAIN_DIR / "sequences.csv", index=False)
    val_df.to_csv(VAL_DIR / "sequences.csv", index=False)
    test_df.to_csv(TEST_DIR / "sequences.csv", index=False)

    save_individual_sequence_csvs(train_df, TRAIN_DIR)
    save_individual_sequence_csvs(val_df, VAL_DIR)
    save_individual_sequence_csvs(test_df, TEST_DIR)

    metadata = {
        "name": DATASET_NAME,
        "radar_type": RADAR_TYPE,
        "sequence_count": SEQUENCE_COUNT,
        "expected_pulses_per_sequence": PULSES_PER_SEQUENCE,
        "error_level": LEVEL,
        "error_parameters": cfg,
        "files": {
            "train": "train/sequences.csv",
            "val": "val/sequences.csv",
            "test": "test/sequences.csv",
            "individual_sequence_files":
                "Each split also contains one seq_XXXX.csv per sequence.",
        },
        "split": {
            "train": {
                "ratio": 0.70,
                "sequence_count": train_df["sequence_id"].nunique(),
                "row_count": len(train_df),
            },
            "val": {
                "ratio": 0.10,
                "sequence_count": val_df["sequence_id"].nunique(),
                "row_count": len(val_df),
            },
            "test": {
                "ratio": 0.20,
                "sequence_count": test_df["sequence_id"].nunique(),
                "row_count": len(test_df),
            },
        },
        "radar_parameters": {
            "radar_id": RADAR_ID,
            **PRI_METADATA,
            "freq_mean_mhz": FREQ_MEAN_MHZ,
            "freq_mode": "constant",
            "poi": cfg["poi"],
            "multipath_prob": cfg["multipath_prob"],
            "aoa_drift_rate": cfg["aoa_drift_rate"],
            "pw_mean_us": PW_MEAN_US,
            "amp_mean_dbm": AMP_MEAN_DBM,
            "aoa_mean_deg": AOA_MEAN_DEG,
        },
        "environment_parameters": {
            "duration_us": PULSES_PER_SEQUENCE * PRI_US,
            "toa_noise_std": cfg["toa_noise_std"],
            "freq_noise_std": cfg["freq_noise_std"],
            "pw_noise_std": cfg["pw_noise_std"],
            "amp_noise_std": cfg["amp_noise_std"],
            "aoa_noise_std": cfg["aoa_noise_std"],
            "global_drop_rate": cfg["global_drop_rate"],
            "noise_rate": cfg["noise_rate"],
            "quantize": cfg["quantize"],
        },
        "units": {
            "TOA": "microseconds",
            "PRI": "microseconds",
            "Frequency": "MHz",
            "PW": "microseconds",
            "Amplitude": "dBm",
            "AOA": "degrees",
        },
    }

    with open(DATASET_DIR / "metadata.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(metadata, f, sort_keys=False)

    print(DATASET_NAME)
    print("Train:", train_df["sequence_id"].nunique(), "seqs,", len(train_df), "rows")
    print("Val:", val_df["sequence_id"].nunique(), "seqs,", len(val_df), "rows")
    print("Test:", test_df["sequence_id"].nunique(), "seqs,", len(test_df), "rows")

single_stagger_1000x128_clean
Train: 700 seqs, 90300 rows
Val: 100 seqs, 12900 rows
Test: 200 seqs, 25800 rows
single_stagger_1000x128_error_lv1
Train: 700 seqs, 89944 rows
Val: 100 seqs, 12853 rows
Test: 200 seqs, 25703 rows
single_stagger_1000x128_error_lv2
Train: 700 seqs, 89851 rows
Val: 100 seqs, 12849 rows
Test: 200 seqs, 25629 rows
single_stagger_1000x128_error_lv3
Train: 700 seqs, 89720 rows
Val: 100 seqs, 12853 rows
Test: 200 seqs, 25633 rows


In [8]:
readme = """# PRI Benchmark Datasets

Synthetic pulse sequences for stable, Gaussian-jitter, and stagger radars.

## New layout

After running the updated generators, datasets are organized as:

    datasets/
      single_stable/
        clean/
        error_lv1/
        error_lv2/
        error_lv3/
      single_jitter/
        clean/
        error_lv1/
        error_lv2/
        error_lv3/
      single_stagger/
        clean/
        error_lv1/
        error_lv2/
        error_lv3/

Each error-level folder contains metadata.yaml and three split folders:
train, val, and test. Each split has a combined sequences.csv and
individual seq_XXXX.csv files.

## Older datasets

Datasets generated before this layout remain in flat folders such as
single_stable_1000x128_error_lv1 and
single_jitter_1000x128_clean. The older single_stable_1000x128
folder may also be present. These folders are not moved or deleted
by the new generators. Read them using their existing paths.

## Sequences and splits

Each error level has 1000 independent sequences: 700 train,
100 validation, and 200 test. Splitting is by sequence_id, so
pulses from one sequence cannot appear in multiple splits.

The 1000x128 designation means 1000 sequences generated over a
duration of 128 mean-PRI intervals. The observed row count can vary,
including in clean jitter or stagger data. Missing pulses reduce the
row count; false pulses and multipath echoes can increase it.

Do not concatenate different sequence_id values as one continuous
radar signal. Sort each sequence by pulse_index before using it.

## Error levels

clean has no added observation errors. The jitter radar still has
intrinsic Gaussian PRI variation, and the stagger radar still has
its repeating PRI pattern. error_lv1, error_lv2, and error_lv3
introduce increasing nonideal effects. See each metadata.yaml for
the exact radar and environment parameters.

## Reading a dataset

From the repository root:

    from pathlib import Path
    import pandas as pd

    dataset = Path("datasets/single_stagger/error_lv1")
    train = pd.read_csv(dataset / "train/sequences.csv")
    val = pd.read_csv(dataset / "val/sequences.csv")
    test = pd.read_csv(dataset / "test/sequences.csv")

    for sequence_id, sequence in train.groupby("sequence_id"):
        sequence = sequence.sort_values("pulse_index")
        # Process this sequence independently.

For an older flat dataset, change only the dataset path, for example:

    dataset = Path("datasets/single_stagger_1000x128_error_lv1")

The PRI column describes the generated pulse's source PRI.
Measured_Delta_TOA is the difference between adjacent observed TOAs;
missing, false, and echo pulses can affect that observed difference.
"""

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
readme_path = DATASET_ROOT / "README.md"
readme_path.write_text(readme, encoding="utf-8")
print(f"Updated: {readme_path}")

Updated: C:\Users\ciftc\OneDrive\Desktop\homeworks\4. year\1\EE 493\repo\Bitirme\datasets\README.md
